# Deep Agents: Building a Travel Planner from Scratch

In this notebook we'll build a **travel planning agent** progressively, layering one Deep Agents capability at a time. By the end we'll have an orchestrator that delegates to three specialist subagents (hotels, flights, activities), uses long-term memory to remember a traveler's preferences across sessions, gates real-money bookings behind human approval, and produces polished itineraries through reusable skills.

**What we'll cover, in order:**

<table style="font-size: 1.15em">
<tr><th>Part</th><th>Concept</th></tr>
<tr><td>0</td><td>Setup &amp; installation</td></tr>
<tr><td>1</td><td>Your first deep agent (the harness)</td></tr>
<tr><td>2</td><td>Adding custom tools (Mock travel data tools (4))</td></tr>
<tr><td>3</td><td>Backends: ephemeral, disk, persistent store, composite routing</td></tr>
<tr><td>4</td><td>Subagents: hotel-search, flight-search, activity-search</td></tr>
<tr><td>5</td><td>Middleware: built-ins + custom tool-call logging</td></tr>
<tr><td>6</td><td>Human-in-the-loop: approval before booking</td></tr>
<tr><td>7</td><td>Long-term memory: semantic, episodic, procedural + per-user namespacing</td></tr>
<tr><td>8</td><td>AGENTS.md &amp; Skills (itinerary, budget, packing list, travel brief)</td></tr>
<tr><td>9</td><td>The complete travel planner</td></tr>
<tr><td>10</td><td>Deploying with the <code>deepagents</code> CLI</td></tr>
<tr><td>11</td><td>Next steps</td></tr>
</table>


## Part 0: Setup & Installation

Install the dependencies and initialize a model. 

In [1]:
# Install required packages
# Run uv sync to install the packages, or:
# !pip install deepagents python-dotenv langchain langgraph


### Initialize your LLM

`init_chat_model` gives you a single uniform interface across providers (OpenAI, Anthropic, Bedrock, Azure). Swap the string and the rest of the notebook still works.


In [2]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env", override=True)

from langchain.chat_models import init_chat_model  # used later for eval model swapping
from utils.models import model  # main agent model lives in utils/models.py -- swap providers / APIM there in one place

import warnings
warnings.filterwarnings('ignore', message='LangSmith now uses UUID v7')


## Part 1: Your First Deep Agent (The Harness)

`create_deep_agent` returns a LangGraph agent pre-wired with three middleware capabilities you'll use in every Deep Agent:

1. **TodoListMiddleware** — gives the agent a `write_todos` tool for planning
2. **FilesystemMiddleware** — gives it `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`
3. **SubAgentMiddleware** — gives it a `task()` tool to delegate to subagents (Part 4)

Plus summarization and prompt-cache middleware that keep long conversations affordable.

Here's the smallest possible travel agent:


In [3]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=model,
    system_prompt=(
        "You are a helpful travel planning assistant. "
        "When referencing file paths, use backtick formatting like `path/file.md` "
        "instead of markdown links."
    ),
)


### Test the built-in filesystem tools

The agent already has a virtual filesystem. Let's ask it to brainstorm trip ideas and save them.


In [4]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Brainstorm 3 long-weekend trip ideas from San Francisco for someone "
            "who loves food and architecture. Save the list to `/trip_ideas.md`, "
            "then list the files you have."
        ),
    }]
})

print(result["messages"][-1].content)


Saved to `/trip_ideas.md`.

Files:
- `/trip_ideas.md`


In [5]:
# THE VIRTUAL FILESYSTEM LIVES IN result["files"]

for path, file_data in result.get("files", {}).items():
    print(f"=== {path} ===")
    print(file_data["content"])
    print()

=== /trip_ideas.md ===
# Long-weekend trip ideas from San Francisco

1. **Mexico City, Mexico**
   - Why go: One of the best food cities in the world, with outstanding street food, markets, and contemporary restaurants.
   - Architecture highlights: Historic center, Palacio de Bellas Artes, Luis Barragán sites, and a strong mix of colonial, Art Deco, and modern design.
   - Long-weekend fit: Nonstop flights from San Francisco make it realistic for a 3–4 day trip.

2. **Chicago, Illinois**
   - Why go: Deep-dish aside, the city has excellent fine dining, neighborhood food scenes, and classic steakhouses.
   - Architecture highlights: River architecture cruises, the Chicago School, Frank Lloyd Wright sites, and iconic skyscrapers.
   - Long-weekend fit: Easy nonstop flights and a compact core for sightseeing over a few days.

3. **Portland, Oregon**
   - Why go: Great coffee, bakeries, farm-to-table restaurants, and inventive casual dining.
   - Architecture highlights: A mix of preserve

### Key Takeaway

A Deep Agent isn't just an LLM — it's a **harness** of middleware that gives the model a planner, a virtual filesystem, and a delegation primitive out of the box. You haven't written any tools yet, and the agent can already plan, write files, and read them back.


## Part 2: Adding Custom Tools

Travel planning needs both **discovery** (what hotels exist?) and **details** (what does it cost?). We'll define two flavors of tools:

- **`search_travel(query, category)`** — a discovery tool scoped to hotel / flight / activity domains. Offline stub for workshop reliability; same `@tool` shape as a real search wrapper.
- **Mock pricing tools** — deterministic Python functions that return fake quotes/rates/availability. They make the rest of the notebook reproducible without burning API quota and give us bookable IDs for the human-in-the-loop section later.

Subagents in Part 4 will each get the relevant subset of these tools.


In [6]:
import hashlib
from typing import Literal
from langchain_core.tools import tool

# --- DISCOVERY TOOL (deterministic offline stub; no network) ---

CATEGORY_DOMAINS = {
    "hotel":    ["booking.com", "hotels.com", "tripadvisor.com"],
    "flight":   ["kayak.com", "google.com/flights", "skyscanner.com"],
    "activity": ["tripadvisor.com", "viator.com", "getyourguide.com"],
}

# City knowledge base. Adding a new city = 4 lines.
CITY_KB = {
    "lisbon": {
        "neighborhoods": ["Alfama", "Chiado", "Bairro Alto"],
        "hotels":        ["Memmo Alfama", "The Vintage Lisbon", "Bairro Alto Hotel"],
        "activities":    ["Belem Tower visit", "Fado dinner in Alfama", "Sintra day trip"],
    },
    "tokyo": {
        "neighborhoods": ["Shibuya", "Ginza", "Asakusa"],
        "hotels":        ["Park Hyatt Tokyo", "Hotel Gracery Shinjuku", "Trunk Hotel"],
        "activities":    ["Tsukiji food tour", "Meiji Shrine visit", "TeamLab Planets"],
    },
    "paris": {
        "neighborhoods": ["Le Marais", "Saint-Germain", "Montmartre"],
        "hotels":        ["Hotel Costes", "Hotel des Grands Boulevards", "Le Bristol Paris"],
        "activities":    ["Louvre evening tour", "Seine river cruise", "Versailles day trip"],
    },
    "barcelona": {
        "neighborhoods": ["Eixample", "Gothic Quarter", "El Born"],
        "hotels":        ["Hotel Casa Fuster", "Cotton House Hotel", "Mercer Hotel Barcelona"],
        "activities":    ["Sagrada Familia tour", "Park Guell visit", "Tapas crawl in El Born"],
    },
    "new york": {
        "neighborhoods": ["SoHo", "West Village", "Williamsburg"],
        "hotels":        ["The Bowery Hotel", "1 Hotel Brooklyn Bridge", "The Greenwich Hotel"],
        "activities":    ["Central Park bike tour", "MoMA visit", "Brooklyn food tour"],
    },
    "nyc": {
        "neighborhoods": ["SoHo", "West Village", "Williamsburg"],
        "hotels":        ["The Bowery Hotel", "1 Hotel Brooklyn Bridge", "The Greenwich Hotel"],
        "activities":    ["Central Park bike tour", "MoMA visit", "Brooklyn food tour"],
    },
}
_DEFAULT_KB = {
    "neighborhoods": ["the city center", "the historic quarter", "the waterfront district"],
    "hotels":        ["a top-rated boutique", "a centrally located mid-range hotel", "a well-reviewed design hotel"],
    "activities":    ["a leading half-day walking tour", "a popular food trail", "a sunset viewpoint visit"],
}

AIRLINES_PREFERRED = ["American Airlines", "Delta", "United", "JetBlue"]


def _seeded_int(key: str, mod: int) -> int:
    return int(hashlib.sha1(key.encode()).hexdigest(), 16) % mod


def _city_for(query: str):
    q = query.lower()
    for city in CITY_KB:
        if city in q:
            return city
    return None


def _hotel_blurbs(query, kb):
    n, h = kb["neighborhoods"], kb["hotels"]
    return [
        (f"Top Boutique Hotels in {query}",
         f"Best central picks this season include {h[0]} in {n[0]} and {h[1]} in {n[1]}. Refundable rates available."),
        (f"{query}: Where to Stay in {n[1]}",
         f"{h[1]} and {h[0]} consistently rank top for travelers. Mid-range to high-end options $250-$400/night."),
        (f"Best Hotel Deals for {query}",
         f"Compare {h[2]} and other central options in {n[2]}. Strong reviews for value, breakfast included."),
    ]


def _flight_blurbs(query):
    return [
        (f"{query}: Best Nonstop Options",
         f"{AIRLINES_PREFERRED[0]} operates daily nonstop service on this route -- usually the cheapest and most direct option. "
         f"{AIRLINES_PREFERRED[1]} and {AIRLINES_PREFERRED[2]} offer 1-stop alternatives."),
        (f"How to Find Deals on {query}",
         f"{AIRLINES_PREFERRED[0]} AAdvantage members consistently see the best fares on this route. "
         f"Tuesday and Wednesday departures typically run 15-25% below weekend prices."),
        (f"{query}: Route Guide",
         f"{', '.join(AIRLINES_PREFERRED[:3])} all serve this route. {AIRLINES_PREFERRED[0]} is the recommended carrier for "
         f"on-time performance and direct schedule. Typical nonstop duration matches industry average."),
    ]


def _activity_blurbs(query, kb):
    a = kb["activities"]
    return [
        (f"Top Things to Do in {query}",
         f"Don't miss {a[0]} -- typically books out 2 weeks ahead. {a[1]} runs daily, small group preferred."),
        (f"{query} Guided Tours and Experiences",
         f"{a[1]} and {a[2]} are visitor favorites. Mix of cultural and food-forward options, half-day to full-day."),
        (f"Local Activities: {query}",
         f"For a fuller day, pair {a[0]} with {a[2]}. Both available with verified guides and instant confirmation."),
    ]


@tool(parse_docstring=True)
def search_travel(query: str, category: Literal["hotel", "flight", "activity"]) -> str:
    """Search for travel info, scoped to a category. Offline stub for workshop reliability.

    Args:
        query: Free-text search (e.g. "boutique hotels Lisbon Alfama October").
        category: One of "hotel", "flight", or "activity". Determines result domains and entity selection.
    """
    domains = CATEGORY_DOMAINS[category]
    city = _city_for(query)
    kb = CITY_KB.get(city, _DEFAULT_KB)
    if category == "hotel":
        blurbs = _hotel_blurbs(query, kb)
    elif category == "flight":
        blurbs = _flight_blurbs(query)
    else:
        blurbs = _activity_blurbs(query, kb)
    base = _seeded_int(f"{query}|{category}", len(blurbs))
    slug = "-".join(query.lower().split()[:4]) or "results"
    items = []
    for i in range(3):
        title, body = blurbs[(base + i) % len(blurbs)]
        domain = domains[(base + i) % len(domains)]
        items.append(f"- {title}\n  https://{domain}/{category}/{slug}\n  {body}")
    return "\n\n".join(items)

# ---   MOCK PRICING/AVAILABILITY TOOLS (deterministic) ---

@tool(parse_docstring=True)
def get_flight_quotes(origin: str, destination: str, date: str) -> str:
    """Get mock flight quotes between two cities on a date.

    Args:
        origin: Origin city or IATA code.
        destination: Destination city or IATA code.
        date: Departure date in YYYY-MM-DD.
    """
    base = (hash((origin, destination, date)) % 400) + 250
    # American Airlines listed first: cheapest fare, nonstop, best schedule.
    return "\n".join([
        f"FL-{base}A | American AA{100+(base+27)%900} | depart 08:15 arrive 11:40 | nonstop | ${base-60}",
        f"FL-{base}B | Delta DL{100+(base+13)%900} | depart 13:05 arrive 17:30 | nonstop | ${base}",
        f"FL-{base}C | United UA{100+base%900}  | depart 21:50 arrive 05:10+1 | 1 stop  | ${base+45}",
    ])

@tool(parse_docstring=True)
def get_hotel_rates(city: str, checkin: str, checkout: str) -> str:
    """Get mock nightly hotel rates for a city and date range.

    Args:
        city: Destination city.
        checkin: Check-in date YYYY-MM-DD.
        checkout: Check-out date YYYY-MM-DD.
    """
    base = (hash((city, checkin)) % 200) + 120
    return "\n".join([
        f"HT-{base}A | The Garden Boutique ({city}) | 4.6star | ${base}/night | refundable",
        f"HT-{base}B | Grand Plaza ({city}) | 4.3star | ${base+90}/night | non-refundable",
        f"HT-{base}C | Riverside Inn ({city}) | 4.8star | ${base+40}/night | refundable",
    ])

@tool(parse_docstring=True)
def get_activity_options(city: str, date: str) -> str:
    """Get mock activity / tour options for a city on a date.

    Args:
        city: Destination city.
        date: Date YYYY-MM-DD.
    """
    base = (hash((city, date)) % 60) + 25
    return "\n".join([
        f"AC-{base}A | Half-day food walking tour | 3h | small group | ${base}",
        f"AC-{base}B | Old-town architecture tour | 2h | private guide | ${base+35}",
        f"AC-{base}C | Sunset harbor cruise | 2h | drinks included | ${base+20}",
    ])


### Add custom tools to our agent

Recreate the agent with all four travel tools. We're not using subagents yet — the orchestrator can call everything directly so you can see how a single agent uses tools before we split it up in Part 4.


In [7]:
agent = create_deep_agent(
    model=model,
    tools=[search_travel, get_flight_quotes, get_hotel_rates, get_activity_options],
    system_prompt=(
        "You are a travel planning assistant. "
        " Use search_travel for discovery "
        "and the get_* tools for prices and availability. Save findings to files. "
        "When referencing file paths, use backtick formatting like `path/file.md`."
    ),
)


In [8]:
# QUICK SMOKE TEST

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "I'm thinking about a 4-day trip to Lisbon in late September. "
            "Pull a few hotel rates and flight quotes from SFO so I can compare."
        ),
    }]
})

print(result["messages"][-1].content)


Comparison for a 4-day Lisbon trip in late September is saved in `​/tmp/lisbon_trip_options.md`.

Assuming Thu 2026-09-24 to Mon 2026-09-28:

**Hotels in Lisbon**
- The Garden Boutique — 4.6★ — **$283/night** — refundable
- Riverside Inn — 4.8★ — **$323/night** — refundable
- Grand Plaza — 4.3★ — **$373/night** — non-refundable

**Flights from SFO to Lisbon**
- American AA712 — nonstop — 08:15 → 11:40 — **$525**
- Delta DL698 — nonstop — 13:05 → 17:30 — **$585**
- United UA685 — 1 stop — 21:50 → 05:10+1 — **$630**

**Notable hotel options found in search**
- Memmo Alfama
- The Vintage Lisbon
- Bairro Alto Hotel

**Best value at a glance**
- Cheapest flight: **American AA712 ($525)**
- Cheapest hotel sample: **The Garden Boutique ($283/night)**
- Best-rated hotel sample: **Riverside Inn (4.8★, $323/night)**

If you want, I can next turn this into 2–3 full trip combinations with estimated total cost.


### Key Takeaway

Custom tools plug into the same agent the same way they would in any LangChain agent — `@tool` and pass them in via `tools=`. The Deep Agents harness adds its built-in tools alongside yours.


## Part 3: Understanding Backends

The virtual filesystem the agent has been writing to is backed by a **Backend**. Deep Agents ship four:

<table style="font-size: 1.35em">
<tr><th>Backend</th><th>Storage</th><th>Persistence</th></tr>
<tr><td><code>StateBackend</code></td><td>LangGraph state</td><td>Per-thread (default)</td></tr>
<tr><td><code>FilesystemBackend</code></td><td>Real disk</td><td>Permanent, on the host</td></tr>
<tr><td><code>StoreBackend</code></td><td>LangGraph <code>BaseStore</code></td><td>Permanent, cross-thread (great for memory)</td></tr>
<tr><td><code>CompositeBackend</code></td><td>Routes paths between the above</td><td>&mdash;</td></tr>
</table>

In this notebook we'll demo `StateBackend` here (the default) and use `CompositeBackend` + `StoreBackend` for memory in Part 7. `FilesystemBackend` (real disk) follows the same pattern — see the docs.

A **checkpointer** lets state survive across `invoke()` calls within the same thread. We'll use `MemorySaver` here; in production you'd swap in `MongoDB` or `Postgres` as a checkpointer.


In [9]:
from langgraph.checkpoint.memory import MemorySaver
from langsmith import uuid7

checkpointer = MemorySaver()

agent = create_deep_agent(
    model=model,
    tools=[search_travel, get_flight_quotes, get_hotel_rates, get_activity_options],
    system_prompt="You are a travel planning assistant.",
    checkpointer=checkpointer,
)

config_a = {"configurable": {"thread_id": str(uuid7())}}    # THREAD_ID IS HOW WE GROUP MESSAGES INTO BACK-AND-FORTH CONVERSATIONS


In [10]:
# WRITE A FILE IN THIS THREAD

agent.invoke(
    {"messages": [{"role": "user", "content": "Save 'Lisbon, Sep 25-29, food + architecture focus' to `/trip_brief.md`."}]},
    config=config_a,
)
files = agent.get_state(config_a).values.get("files", {})
print("Files in thread A:", list(files))


Files in thread A: ['/trip_brief.md']


In [11]:
# IN THE SAME THREAD, THE FILE PERSISTS ACROSS INVOCATIONS

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Read `/trip_brief.md` and tell me what's in it."}]},
    config=config_a,
)
print(result["messages"][-1].content)


`/trip_brief.md` contains:

Lisbon, Sep 25-29, food + architecture focus


In [12]:
# IN A NEW THREAD, THE FILE IS GONE -- StateBackend IS PER-THREAD

config_b = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "List the files you have."}]},
    config=config_b,
)
print(result["messages"][-1].content)


No files are visible in `/` right now.


### Key Takeaway

Backends are pluggable. The same agent code can write to ephemeral state (`StateBackend`, per-thread, great for dev/testing) or a persistent store (`StoreBackend`, cross-thread, great for long-term memory).

We wire `StoreBackend` into the full agent in **Part 7** so the traveler's seat preference, dietary needs, and loyalty numbers persist across sessions.

## Part 4: Adding Subagents (Hotel, Flight, Activity)

A single agent juggling 7+ tools and a multi-step trip can lose focus. Subagents fix this by giving each specialist its own:

- **Isolated context window** — the orchestrator doesn't see the subagent's tool outputs, only its final report
- **Smaller toolset** — easier for the model to choose
- **Focused system prompt** — narrowly scoped instructions

We'll define **three** subagents — `hotel-search`, `flight-search`, `activity-search` — and an orchestrator that delegates to all three (often in parallel) and synthesizes the results into an itinerary. We can also use a smaller model for a subagent like gpt-5.4-mini.


In [13]:
from datetime import datetime
from deepagents import SubAgent

today = datetime.now().strftime("%Y-%m-%d")

hotel_subagent: SubAgent = {
    "name": "hotel-search",
    "description": (
        "Find lodging options for a city and date range. "
        "Provide city, check-in, check-out, and any preferences (budget, neighborhood, amenities)."
    ),
    "system_prompt": (
        f"You are a hotel research specialist. Today is {today}.\n\n"
        "1. Use `search_travel(category='hotel')` to discover options (max 2 calls).\n"
        "2. Use `get_hotel_rates` for concrete pricing.\n"
        "3. Return a short markdown list: 3-5 options with hotel ID, name, price/night, and a one-line note.\n"
        "Do NOT recommend a single winner -- return options."
    ),
    "tools": [search_travel, get_hotel_rates],
}

flight_subagent: SubAgent = {
    "name": "flight-search",
    "description": (
        "Find flight options between two cities for a given date. "
        "Provide origin, destination, and date."
    ),
    "system_prompt": (
        f"You are a flight research specialist. Today is {today}.\n\n"
        "1. Optionally use `search_travel(category='flight')` for context (max 1 call).\n"
        "2. Use `get_flight_quotes` for concrete itineraries.\n"
        "3. Return a markdown list: 3 options with flight ID, airline, times, stops, price."
    ),
    "tools": [search_travel, get_flight_quotes],
}

activity_subagent: SubAgent = {
    "name": "activity-search",
    "description": (
        "Find activities, tours, and attractions for a city on a date. "
        "Provide city, date, and any interests (food, history, nightlife, outdoors)."
    ),
    "system_prompt": (
        f"You are a local activities specialist. Today is {today}.\n\n"
        "1. Use `search_travel(category='activity')` to discover what's notable (max 2 calls).\n"
        "2. Use `get_activity_options` for bookable tours and prices.\n"
        "3. Return a markdown list: 4-6 activities with ID, name, duration, price, and a one-line description."
    ),
    "tools": [search_travel, get_activity_options],
}


In [14]:
ORCHESTRATOR_PROMPT = """You are a travel planning coordinator.

When asked to plan a trip:
1. Use `write_todos` to outline your plan.
2. Delegate research to the specialist subagents using the `task()` tool, IN PARALLEL where possible:
   - `hotel-search` for lodging
   - `flight-search` for flights
   - `activity-search` for things to do
3. Synthesize the subagent reports into a single itinerary saved to `/itinerary.md`.

When referencing file paths, use backtick formatting like `path/file.md`.
"""

orchestrator = create_deep_agent(
    model=model,
    system_prompt=ORCHESTRATOR_PROMPT,
    subagents=[hotel_subagent, flight_subagent, activity_subagent],
    checkpointer=MemorySaver(),
)

result = orchestrator.invoke(
    {"messages": [{"role": "user", "content": (
        "Plan 3 days in Tokyo, starting on June 27, mid-budget, food-focused. I'll fly from SFO."
    )}]},
    config={"configurable": {"thread_id": str(uuid7())}},
)
print(result["messages"][-1].content)

# PLANNING SIGNAL: the todos the orchestrator wrote as it broke the work down.
from langchain_core.messages import AIMessage
print("\n--- Planner todos ---")
for msg in result["messages"]:
    if not isinstance(msg, AIMessage):
        continue
    for tc in msg.tool_calls or []:
        if tc["name"] != "write_todos":
            continue
        for i, todo in enumerate(tc["args"].get("todos", []), 1):
            marker = {"completed": "✓", "in_progress": "▶", "pending": "·"}.get(todo.get("status", "?"), "·")
            print(f"  {marker} {i}. {todo.get('content', todo)}")

Saved the itinerary to `/itinerary.md`.

Top picks:
- Flight: **American AA653 nonstop** from SFO, approx. **$466**
- Best hotel areas: **Asakusa** or **Ueno**
- Best hotel fit: **The Garden Boutique** for value, **Riverside Inn** for slightly more comfort

3-day structure:
- **Day 1:** Tsukiji → Ginza → Yurakucho
- **Day 2:** Asakusa → Kappabashi → Ueno
- **Day 3:** Harajuku → Shibuya → Shinjuku

If you want, I can also make a tighter version with:
1. specific restaurants for each meal, or
2. a version optimized around **Haneda vs Narita arrival**.

--- Planner todos ---
  ▶ 1. Confirm core trip assumptions and create planning structure for 3 days in Tokyo starting June 27 from SFO
  · 2. Research flight options from SFO to Tokyo
  · 3. Research mid-budget hotel options in Tokyo
  · 4. Research food-focused activities and dining ideas for 3 days in Tokyo
  · 5. Synthesize findings into a day-by-day itinerary and save to `/itinerary.md`
  ✓ 1. Confirm core trip assumptions and create p

**NOTE:** Each subagent runs in an isolated context. The orchestrator only sees the *final report* each subagent returns -- not the dozen intermediate tool calls. That's how you scale Deep Agents to long, complex tasks without blowing the context window.


### Key Takeaway

Subagents are isolated, specialized agents you delegate to via the `task()` tool. They have their own tools, prompts, and context. The orchestrator stays high-level: plan, delegate, synthesize, save.


## Part 5: Middleware Deep Dive

Every Deep Agent is built from middleware. The defaults we've been using:

<table style="font-size: 1.15em">
<tr><th>Middleware</th><th>What it adds</th></tr>
<tr><td><code>TodoListMiddleware</code></td><td><code>write_todos</code> tool for planning</td></tr>
<tr><td><code>FilesystemMiddleware</code></td><td><code>ls</code>, <code>read_file</code>, <code>write_file</code>, <code>edit_file</code>, <code>glob</code>, <code>grep</code>, <code>execute</code></td></tr>
<tr><td><code>SubAgentMiddleware</code></td><td><code>task()</code> tool for delegation</td></tr>
<tr><td><code>SummarizationMiddleware</code></td><td>Auto-compacts conversation when it gets long</td></tr>
<tr><td><code>AnthropicPromptCachingMiddleware</code></td><td>Cache prefixes for Anthropic models</td></tr>
</table>

You can add your own. Let's see todos in action first, then build a custom **tool-call logger** so we can watch the orchestrator + subagents fanning out.


### Custom Middleware: Logging Tool Calls

`@wrap_tool_call` lets you intercept every tool invocation. We'll log the tool name and a short preview of the args, then call `handler(request)` to run the tool unchanged. This is invaluable for understanding what your agent is actually doing -- especially with parallel subagents.


In [15]:
from langchain.agents.middleware import wrap_tool_call

@wrap_tool_call
def log_tool_calls(request, handler):
    """Print every tool call before and after execution."""
    name = request.tool_call["name"]
    args = request.tool_call.get("args", {})
    # For task() calls, surface which subagent is being invoked
    if name == "task":
        sub = args.get("subagent_type", "?")
        desc = (args.get("description") or "")[:80]
        print(f"\u27a4  task -> {sub}: {desc}...")
    else:
        preview = ", ".join(f"{k}={str(v)[:40]}" for k, v in args.items())
        print(f"\u27a4  {name}({preview})")
    result = handler(request)
    print(f"\u2713  {name} done")
    return result


In [16]:
# RECREATE THE ORCHESTRATOR WITH OUR LOGGING MIDDLEWARE APPENDED

logged_orchestrator = create_deep_agent(
    model=model,
    system_prompt=ORCHESTRATOR_PROMPT,
    subagents=[hotel_subagent, flight_subagent, activity_subagent],
    middleware=[log_tool_calls],
    checkpointer=MemorySaver(),
)

result = logged_orchestrator.invoke(
    {"messages": [{"role": "user", "content": (
        "Plan a quick 3-day trip to Barcelona in mid-October from SFO. "
        "I want a couple of hotel and flight options plus one activity per day."
    )}]},
    config={"configurable": {"thread_id": str(uuid7())}},
)
print("\n--- FINAL RESPONSE ---")
print(result["messages"][-1].content)


➤  write_todos(todos=[{'content': 'Gather trip assumptions an)
✓  write_todos done
➤  task -> flight-search: Find 2-3 practical round-trip flight options for a quick 3-day leisure trip from...
➤  task -> hotel-search: Find 2-3 hotel options in Barcelona for a 3-night stay in mid-October 2025, assu...
➤  task -> activity-search: Find good Barcelona activities for a 3-day trip in mid-October 2025. Return 1 re...
✓  task done
✓  task done
✓  task done
➤  write_todos(todos=[{'content': 'Gather trip assumptions an)
✓  write_todos done
➤  write_file(file_path=/itinerary.md, content=# Quick 3-Day Barcelona Trip from SFO

#)
✓  write_file done
➤  write_todos(todos=[{'content': 'Gather trip assumptions an)
✓  write_todos done

--- FINAL RESPONSE ---
Saved the itinerary to `/itinerary.md`.

Highlights:
- Flights: 2 strong nonstop options
  - American: about **$807 round-trip**
  - Delta: about **$927 round-trip**
- Hotels:
  - **The Garden Boutique**: about **$139/night**
  - **Grand Plaza**: ab

### Key Takeaway

Middleware is the extension point of Deep Agents.

The built-ins give you planning, files, and delegation; `@wrap_tool_call` lets you bolt on observability, retries, caching, or arg-rewriting in a few lines. The same hook works for orchestrator and subagents alike — you'll see `task()` calls fan out as each subagent is called.


## Part 6: Human-in-the-Loop

Some tool calls shouldn't run without your blessing — for a travel agent, **bookings**. Deep Agents ships HITL via `interrupt_on={tool_name: True}`. When the agent tries to call a gated tool, the graph pauses, the call surfaces in the result as `__interrupt__`, and you resume with `Command(resume=...)` after deciding `approve` / `reject` / `edit`.

Let's add bookable mock tools and gate them.


You can also customize *which* decisions are allowed per tool via `InterruptOnConfig` (`approve`/`edit`/`reject`/`response`) — useful if you want to allow editing a quote before approving it. We'll use the simplest form (`True` = always pause, all decisions allowed) here.


In [17]:
@tool(parse_docstring=True)
def book_flight(flight_id: str) -> str:
    """Book a flight by its ID. PRETEND this charges a credit card.

    Args:
        flight_id: A flight ID like 'FL-321A' returned by get_flight_quotes.
    """
    return f"BOOKED {flight_id} -- confirmation #CONF-{abs(hash(flight_id)) % 100000:05d}"

@tool(parse_docstring=True)
def book_hotel(hotel_id: str) -> str:
    """Book a hotel by its ID. PRETEND this charges a credit card.

    Args:
        hotel_id: A hotel ID like 'HT-184B' returned by get_hotel_rates.
    """
    return f"BOOKED {hotel_id} -- confirmation #CONF-{abs(hash(hotel_id)) % 100000:05d}"

hitl_checkpointer = MemorySaver()

agent_with_hitl = create_deep_agent(
    model=model,
    tools=[get_flight_quotes, get_hotel_rates, book_flight, book_hotel],
    system_prompt=(
        "You are a travel concierge. You may research and quote freely, "
        "but call book_flight / book_hotel only when the user explicitly asks to book."
    ),
    interrupt_on={
        "book_flight": True,
        "book_hotel": True,
    },
    checkpointer=hitl_checkpointer,
)


#### Let's give it a try

Ask the agent to find a flight and book the cheapest. It will pause before the booking call.


In [18]:
from langgraph.types import Command
import json

hitl_config = {"configurable": {"thread_id": str(uuid7())}}

result = agent_with_hitl.invoke(
    {"messages": [{"role": "user", "content": (
        "Find me a flight from SFO to LIS on 2026-09-23, then BOOK the cheapest one."
    )}]},
    config=hitl_config,
)

if result.get("__interrupt__"):
    interrupt = result["__interrupt__"][0]
    print("=== AGENT PAUSED FOR APPROVAL ===")
    print(json.dumps(interrupt.value, indent=2, default=str))
else:
    print("No interrupt -- agent didn't try to book.")
    print(result["messages"][-1].content)


=== AGENT PAUSED FOR APPROVAL ===
{
  "action_requests": [
    {
      "name": "book_flight",
      "args": {
        "flight_id": "FL-599A"
      },
      "description": "Tool execution requires approval\n\nTool: book_flight\nArgs: {'flight_id': 'FL-599A'}"
    }
  ],
  "review_configs": [
    {
      "action_name": "book_flight",
      "allowed_decisions": [
        "approve",
        "edit",
        "reject"
      ]
    }
  ]
}


#### Resume with approval

Pass `Command(resume={"decisions": [{"type": "approve"}]})` to continue. Use `"reject"` to refuse.


In [19]:
if result.get("__interrupt__"):
    result = agent_with_hitl.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=hitl_config,
    )
    print(result["messages"][-1].content)


Booked: FL-599A — American AA726  
SFO → LIS on 2026-09-23  
Depart 08:15, arrive 11:40, nonstop  
Price: $539  
Confirmation: CONF-53027


### Key Takeaway

`interrupt_on` lets you put a **human gate** in front of any tool. Combined with the checkpointer, the agent's full state survives the pause — you can resume hours later, on a different machine, after approval lands in a Slack message or a PR review. Booking, sending email, deleting data: all good candidates.


## Part 7: Long-Term Memory

`StateBackend` is per-thread. For an agent that should remember **across conversations** — the traveler's seat preference, dietary needs, loyalty numbers — we need the `StoreBackend`, which writes to LangGraph's persistent `BaseStore`.

Pattern: route `/memories/` to a `StoreBackend`, leave everything else in `StateBackend`. The agent uses the same `read_file`/`write_file` tools either way; the backend handles persistence.


In [20]:
from langgraph.config import get_config
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StateBackend, StoreBackend, CompositeBackend

def _current_user_id() -> str:
    cfg = get_config() or {}
    # `or "anonymous"` handles both missing AND empty-string user_id (Studio sometimes passes user_id="")
    return cfg.get("configurable", {}).get("user_id") or "anonymous"

def per_user_memory_backend(rt):
    # Important: get_config() is called inside the namespace lambda so that
    # user_id is resolved per store operation (i.e. per request), not frozen
    # at backend construction.
    return CompositeBackend(
        default=StateBackend(),
        routes={
            "/memories/user/":   StoreBackend(namespace=lambda ctx: ("user", _current_user_id(), "filesystem")),
            "/memories/shared/": StoreBackend(namespace=lambda ctx: ("shared", "filesystem")),
        },
    )

multiuser_store = InMemoryStore()

multiuser_agent = create_deep_agent(
    model=model,
    system_prompt=(
        "You are a travel concierge serving many travelers. "
        "Save personal preferences to `/memories/user/preferences.md`. "
        "Save universally-applicable rules (visa info, supplier blacklist) to `/memories/shared/rules.md`."
    ),
    backend=per_user_memory_backend,
    checkpointer=MemorySaver(),
    store=multiuser_store,
)

In [21]:
# ALICE WRITES _BOTH_ A PRIVATE _AND_ A SHARED NOTE

alice_cfg = {"configurable": {"thread_id": str(uuid7()), "user_id": "alice"}}
multiuser_agent.invoke(
    {"messages": [{"role": "user", "content": (
        "I'm Alice. Save to my personal preferences: window seat, vegetarian, Marriott Bonvoy #1234. "
        "Also save to shared rules: 'US passport holders need ETIAS for Schengen starting 2025'."
    )}]},
    config=alice_cfg,
)
print("Alice saved both files.")

Alice saved both files.


In [22]:
# BOB (DIFFERENT USER_ID) CHECKS WHAT _HE_ CAN SEE

bob_cfg = {"configurable": {"thread_id": str(uuid7()), "user_id": "bob"}}
result = multiuser_agent.invoke(
    {"messages": [{"role": "user", "content": (
        "I'm Bob. List everything you have under `/memories/user/` AND under `/memories/shared/`."
    )}]},
    config=bob_cfg,
)
print(result["messages"][-1].content)
print("\nExpected: Bob sees the SHARED ETIAS rule but NOT Alice's preferences.")

`/memories/user/`
- empty

`/memories/shared/`
- `/memories/shared/rules.md`

Expected: Bob sees the SHARED ETIAS rule but NOT Alice's preferences.


### Key Takeaway

The `namespace` callable on `StoreBackend` is your isolation primitive. Tuple components like `("user", user_id, ...)` or `("shared", ...)` cleanly partition the store. Alice's seat preferences stay private; the ETIAS rule is global. The agent sees the same paths regardless — only the underlying namespace differs.


## Part 8: AGENTS.md & Skills

So far we've put the agent's identity inline in `system_prompt`. That works, but it doesn't scale: as your agent grows, its instructions become a sprawling string, and there's no way for it to **learn** by editing them.

Two patterns fix this:

- **`AGENTS.md`** — a markdown file the agent loads at startup as part of its identity. The agent can `edit_file` it during a conversation to record lessons learned. Pass via `memory=["/AGENTS.md"]`.
- **Skills (`SKILL.md`)** — small markdown files describing *how to do a specific thing* (write an itinerary in our standard format, produce a budget summary table, etc.). The agent only loads a skill when it needs it (progressive disclosure). Pass via `skills=["/skills/"]`.

Let's define an `AGENTS.md` for our travel planner and four skills.


In [23]:
from deepagents.backends.utils import create_file_data

AGENTS_MD = """# Travel Planner

You are an expert travel concierge. You research, quote, and produce polished trip plans.

## Workflow

1. **Understand**   -- ask 2-3 clarifying questions if the request is vague (dates, budget, style, party size)
2. **Load skills**  -- for every deliverable the user asks for (itinerary, budget summary, packing list, travel brief), `read_file` the matching `/skills/<name>/SKILL.md` BEFORE step 3. This is mandatory, even if the format feels obvious -- the SKILL.md is the only authoritative source for our exact format
3. **Plan**         -- use `write_todos` to outline the work
4. **Delegate**     -- use `task()` to call the hotel-search, flight-search, and activity-search subagents IN PARALLEL where possible
5. **Synthesize**   -- combine subagent reports into `/itinerary.md` and `/budget.md` in one pass; do NOT pause for confirmation between research and synthesis
6. **Remember**     -- save lasting traveler facts to `/memories/user/preferences.md`

## Rules

- After step 1's clarifying questions, do not ask for any further user confirmation. Make a reasonable choice from the subagents' options (cheapest, best-rated, or best fit for stated style) and proceed straight through to writing the deliverable file

- Booking tools (book_flight, book_hotel) require explicit user approval
- Cite hotel/flight IDs from quotes so the user can ask you to book them
- When referencing file paths, use backtick formatting like `path/file.md`
"""

### Now let's define our agent's Skills

Each skill lives at `/skills/<name>/SKILL.md`. The agent will discover them via `glob`/`ls` and read the relevant one when the task calls for it.


In [24]:
ITINERARY_SKILL = """---
name: itinerary-format
description: Produce a day-by-day travel itinerary using our standard structure.
---

# Itinerary Format

Use this exact structure when writing `/itinerary.md`:

```
# <Destination> -- <Trip Dates>

## Trip Overview
- Travelers: ...
- Style: ...
- Budget: ...

## Day 1 -- <Date> (<Day of week>)
- Morning:   ...
- Afternoon: ...
- Evening:   ...
- Lodging:   ...

## Day 2 -- <Date> ...
(repeat per day)

## Logistics
- Flights:    ...
- Transit:    ...
- Currency:   ...
```

Always include time blocks (Morning/Afternoon/Evening). Always end with a Logistics section.
"""

BUDGET_SKILL = """---
name: budget-summary
description: Produce a cost-breakdown table for a trip.
---

# Budget Summary Format

Use this exact structure when writing `/budget.md`:

```
# Budget -- <Destination>, <Dates>

| Category    | Item                  | Cost    |
|-------------|-----------------------|---------|
| Flights     | <airline + route>     | $...    |
| Lodging     | <hotel x N nights>    | $...    |
| Activities  | <tour name>           | $...    |
| Food        | <est. per day x days> | $...    |
| Buffer 10%  |                       | $...    |
| **TOTAL**   |                       | **$...**|
```

Always include a 10% buffer line. Total in bold.
"""

PACKING_SKILL = """---
name: packing-list
description: Produce a climate- and activity-aware packing checklist.
---

# Packing List Format

Use this structure when writing `/packing_list.md`:

```
# Packing List -- <Destination>, <Dates>

## Documents
- [ ] Passport (+ photocopy)
- [ ] Visa / ETIAS confirmation if applicable
- [ ] Travel insurance card

## Climate-appropriate clothing
- [ ] (specific items based on forecast)

## Activity-specific gear
- [ ] (e.g. hiking shoes if hiking, swim gear if beach)

## Tech
- [ ] Plug adapter (specify type)
- [ ] Chargers
```
"""

BRIEF_SKILL = """---
name: travel-brief
description: Produce a one-page traveler brief with logistics essentials.
---

# Travel Brief Format

Use this structure when writing `/travel_brief.md`:

```
# Travel Brief -- <Destination>

- **Currency:**    <code> (~<rate vs USD>)
- **Plug type:**   <letter> (<voltage>)
- **Visa:**        <required? on arrival? ETIAS?>
- **Emergency:**   <local 911-equivalent>
- **Time zone:**   <UTC offset>
- **Tipping:**     <norms>

## Key phrases
- Hello: ...
- Thank you: ...
- Where is the bathroom?: ...
```
"""

### Put It All Together (AGENTS.md + Skills)

We seed the virtual filesystem with the AGENTS.md and the four skill files using `create_file_data`, then point the agent at them via `memory=` and `skills=`.


In [25]:
seed_files = {
    "/AGENTS.md": create_file_data(AGENTS_MD),
    "/skills/itinerary-format/SKILL.md": create_file_data(ITINERARY_SKILL),
    "/skills/budget-summary/SKILL.md":   create_file_data(BUDGET_SKILL),
    "/skills/packing-list/SKILL.md":     create_file_data(PACKING_SKILL),
    "/skills/travel-brief/SKILL.md":     create_file_data(BRIEF_SKILL),
}

# Track skill reads from anywhere (orchestrator OR subagent), since the
# orchestrator's `result["messages"]` doesn't include subagent tool calls.
tracked_skill_reads: list[str] = []

@wrap_tool_call
def track_skill_reads(request, handler):
    if request.tool_call["name"] == "read_file":
        path = request.tool_call.get("args", {}).get("file_path", "")
        if "skills/" in path and path.endswith("SKILL.md"):
            tracked_skill_reads.append(path)
    return handler(request)

skill_agent = create_deep_agent(
    model=model,
    subagents=[hotel_subagent, flight_subagent, activity_subagent],
    memory=["/AGENTS.md"],
    skills=["/skills/"],
    middleware=[log_tool_calls, track_skill_reads],
    checkpointer=MemorySaver(),
)

def show_file(result, path):
    file = result.get("files", {}).get(path)
    if not file:
        print(f"(no {path} found)")
        return
    print(f"\n----- {path} -----")
    print("".join(file["content"]))


In [26]:
# TEST 1: ASK FOR AN ITINERARY -- THE AGENT SHOULD LOAD THE ITINERARY-FORMAT SKILL

tracked_skill_reads.clear()

result = skill_agent.invoke(
    {"messages": [{"role": "user", "content": (
        "Before producing any output, call `read_file` on `/skills/itinerary-format/SKILL.md` to load the exact format. "
        "Then plan a 3-day Porto trip from SFO, October 14-16, solo traveler, "
        "mid-budget (~$1500 total), food and architecture focused, "
        "and produce the itinerary in `/itinerary.md` following the loaded skill exactly."
    )}], "files": seed_files},
    config={"configurable": {"thread_id": str(uuid7())}},
)

print(f"\nSkills loaded during run: {tracked_skill_reads or '(none)'}\n")
print(result["messages"][-1].content)

show_file(result, "/itinerary.md")


➤  read_file(file_path=/skills/itinerary-format/SKILL.md)
✓  read_file done
➤  write_todos(todos=[{'content': 'Research trip parameters a)
✓  write_todos done
➤  task -> flight-search: Find reasonable round-trip flight options for a solo traveler from SFO to Porto ...
➤  task -> activity-search: Find food- and architecture-focused activities and attractions in Porto for Octo...
➤  task -> hotel-search: Find 3 mid-budget hotel options in Porto for a solo traveler for 2 nights coveri...
✓  task done
✓  task done
✓  task done
➤  write_todos(todos=[{'content': 'Research trip parameters a)
✓  write_todos done
➤  write_file(file_path=/itinerary.md, content=# Porto -- October 14-16

## Trip Overvi)
✓  write_file done
➤  read_file(file_path=/itinerary.md)
✓  read_file done
➤  write_todos(todos=[{'content': 'Research trip parameters a)
✓  write_todos done

Skills loaded during run: ['/skills/itinerary-format/SKILL.md']

Done. The itinerary was written to `/itinerary.md` in the exact skill forma

In [27]:
# TEST 2: ASK FOR A BUDGET SUMMARY -- THE AGENT SHOULD LOAD THE BUDGET-SUMMARY SKILL ON DEMAND

tracked_skill_reads.clear()

result = skill_agent.invoke(
    {"messages": [{"role": "user", "content": (
        "Before producing any output, call `read_file` on `/skills/budget-summary/SKILL.md` to load the exact format. "
        "Then plan a 4-day Marrakech trip from SFO, October 22-25, two travelers, "
        "mid-budget (~$2500 total), cultural and food focused, "
        "and produce a budget summary in `/budget.md` following the loaded skill exactly."
    )}], "files": seed_files},
    config={"configurable": {"thread_id": str(uuid7())}},
)

print(f"\nSkills loaded during run: {tracked_skill_reads or '(none)'}\n")
print(result["messages"][-1].content)
show_file(result, "/budget.md")



➤  read_file(file_path=/skills/budget-summary/SKILL.md)
✓  read_file done
➤  write_todos(todos=[{'content': 'Research likely mid-budget)
✓  write_todos done
➤  task -> flight-search: Find roundtrip flight options for 2 travelers from SFO to Marrakech (RAK) for a ...
➤  task -> hotel-search: Find mid-budget lodging options in Marrakech for 2 travelers checking in 2026-10...
➤  task -> activity-search: Find cultural and food-focused paid activities in Marrakech suitable for 2 trave...
✓  task done
✓  task done
✓  task done
➤  write_todos(todos=[{'content': 'Research likely mid-budget)
✓  write_todos done
➤  write_file(file_path=/budget.md, content=# Budget -- Marrakech, October 22-25

| )
✓  write_file done
➤  write_todos(todos=[{'content': 'Research likely mid-budget)
✓  write_todos done

Skills loaded during run: ['/skills/budget-summary/SKILL.md']

Created `\/budget.md` in the required format.

Note: the estimated total is **$2,682**, which is slightly above the stated **~$2,500** mid

### Key Takeaway

`AGENTS.md` is your agent's persistent identity — it can be edited (by you, or by the agent itself) and survives across runs. **Skills** are bite-sized, on-demand instructions for specific output formats. Together they replace ever-growing system prompts with a clean, file-based knowledge layer.


## Part 9: The Complete Travel Planner

Time to assemble everything we've built:

- 3 specialist subagents (hotel / flight / activity)
- Custom `search_travel` + 3 mock pricing tools
- 2 booking tools gated by HITL
- `AGENTS.md` for identity + 4 skills for output formats
- Tool-call logging middleware
- `CompositeBackend` routing per-user `/memories/` to a `StoreBackend`
- `MemorySaver` checkpointer

Then run a real trip planning task end-to-end.


In [28]:
final_store = InMemoryStore()
final_checkpointer = MemorySaver()

travel_planner = create_deep_agent(
    model=model,
    tools=[book_flight, book_hotel],   # BOOKINGS LIVE ON THE ORCHESTRATOR (GATED BY HUMAN-IN-THE-LOOP CONTROLS)
    subagents=[hotel_subagent, flight_subagent, activity_subagent],
    memory=["/AGENTS.md"],
    skills=["/skills/"],
    middleware=[log_tool_calls],
    backend=per_user_memory_backend,
    checkpointer=final_checkpointer,
    store=final_store,
    interrupt_on={"book_flight": True, "book_hotel": True},
)


In [29]:
# RUN AN END-TO-END TRIP PLANNING REQUEST

trip_thread = {"configurable": {"thread_id": str(uuid7()), "user_id": "alice"}}

result = travel_planner.invoke(
    {"messages": [{"role": "user", "content": (
        "I'm Alice. Plan 7 days in Lisbon for two travelers, departing from SFO in late September. "
        "Budget around $3,000 for flights + lodging combined. "
        "We're foodies and we'd like a couple of light hikes near the city; "
        "book a centrally located mid-range boutique hotel. "
        "Make reasonable choices on flights and lodging without asking for confirmation. "
        "Before producing output, call `read_file` on `/skills/itinerary-format/SKILL.md` and "
        "`/skills/budget-summary/SKILL.md`, then produce both deliverables in our standard formats. "
        "Save my preferences (vegetarian, window seat) for next time."
    )}], "files": seed_files},
    config=trip_thread,
)

print("\n--- ALL MESSAGES ---")
for msg in result["messages"]:
    msg.pretty_print()


print("\n--- FINAL RESPONSE ---")
print(result["messages"][-1].content)


➤  edit_file(file_path=/memories/user/preferences.md, old_string=# User Preferences
, new_string=# User Preferences
- Name: Alice
- Dieta, replace_all=False)
✓  edit_file done
➤  ls(path=/memories)
✓  ls done
➤  write_file(file_path=/memories/user/preferences.md, content=# User Preferences
- Name: Alice
- Dieta)
✓  write_file done
➤  read_file(file_path=/skills/itinerary-format/SKILL.md)
✓  read_file done
➤  read_file(file_path=/skills/budget-summary/SKILL.md)
✓  read_file done
➤  write_todos(todos=[{'content': 'Research flight options fr)
✓  write_todos done
➤  task -> flight-search: Find reasonable round-trip flight options for 2 travelers departing SFO to Lisbo...
➤  task -> hotel-search: Find centrally located mid-range boutique hotel options in Lisbon for 2 traveler...
✓  task done
✓  task done
➤  write_todos(todos=[{'content': 'Research flight options fr)
✓  write_todos done
➤  write_file(file_path=/itinerary.md, content=# Lisbon -- Sep 25-Oct 2, 2026

## Trip )
✓  write_file don

In [30]:
# INSPECT WHAT THE PLANNER PRODUCED

state = travel_planner.get_state(trip_thread)
files = state.values.get("files", {})
for path in sorted(files.keys()):
    if path.startswith(("/AGENTS", "/skills/")):
        continue   # skip the seeded files
    print(f"\n=== {path} ===")
    print(files[path]["content"])



=== /budget.md ===
# Budget -- Lisbon, Sep 25-Oct 2, 2026

| Category    | Item                  | Cost    |
|-------------|-----------------------|---------|
| Flights     | American/Delta SFO-LIS round-trip for 2 (`FL-574A` + `FL-282B`) | $796    |
| Lodging     | The Garden Boutique x 7 nights (`HT-237A`)    | $1,659    |
| Activities  | Two light-hike day trips + local transit estimate           | $120    |
| Food        | est. $100 per day x 7 days | $700    |
| Buffer 10%  |                       | $328    |
| **TOTAL**   |                       | **$3,603**|


=== /itinerary.md ===
# Lisbon -- Sep 25-Oct 2, 2026

## Trip Overview
- Travelers: 2
- Style: Food-focused city break with central boutique lodging, neighborhood wandering, markets, miradouros, and two light hikes near the city
- Budget: Around $3,000 for flights + lodging combined

## Day 1 -- Sep 25, 2026 (Friday)
- Morning: Arrive in Lisbon, transfer to central hotel, check in or drop bags, then take an easy orientati

### Key Takeaway

A production-style Deep Agent is just composition: a base harness (`create_deep_agent`) plus the layers we built one at a time — subagents, custom tools, memory backends, HITL gates, identity files, skills, observability middleware. Each layer is small and independently testable. The travel domain happens to be a great fit, but the same recipe works for any orchestrator-plus-specialists workload.


## Part 10: Evaluations

We've built a working travel planner. Now: how do we know whether it works *consistently*, and how does model choice affect quality, cost, and latency?

We'll use **LangSmith evaluations** to:

1. Build a small dataset of travel-research prompts
2. Define evaluators that check the agent's **tool trajectory**, **constraints** (no surprise bookings), **efficiency**, and **response quality** (LLM-as-judge)
3. Run the same dataset against **`gpt-5.4`** and **`gpt-4.1-mini`** and compare results side-by-side in the LangSmith UI

### 10.1 Build a dataset

Each example pairs a user query with expected **subagent delegations** (the orchestrator's observable trajectory) and a list of tools that should **never** be called for a research-only request.

In [31]:
from datetime import datetime
from langsmith import Client

client = Client()
dataset_name = f"travel-planner-eval-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

examples = [
    {
        "inputs": {"query": "4-day Lisbon trip late September from SFO -- pull hotel rates and flight quotes so I can compare."},
        "outputs": {
            "expected_tools": ["hotel-search", "flight-search"],
            "should_not_call": ["book_flight", "book_hotel"],
        },
    },
    {
        "inputs": {"query": "What activities are worth doing in Tokyo on 2026-10-15? Mid-range, half-day options."},
        "outputs": {
            "expected_tools": ["activity-search"],
            "should_not_call": ["book_flight", "book_hotel"],
        },
    },
    {
        "inputs": {"query": "Compare nonstop flights NYC to Paris for 2026-11-10. Give me 3 options with prices."},
        "outputs": {
            "expected_tools": ["flight-search"],
            "should_not_call": ["book_flight", "book_hotel"],
        },
    },
    {
        "inputs": {"query": "Plan a week in Barcelona starting 2026-12-01: I need hotels, flights from SFO, and things to do."},
        "outputs": {
            "expected_tools": ["hotel-search", "flight-search", "activity-search"],
            "should_not_call": ["book_flight", "book_hotel"],
        },
    },
    {
        "inputs": {"query": "Find a centrally-located boutique hotel in Lisbon for the last weekend of October."},
        "outputs": {
            "expected_tools": ["hotel-search"],
            "should_not_call": ["book_flight", "book_hotel"],
        },
    },
]

dataset = client.create_dataset(dataset_name, description="Travel-planner research queries with expected subagent delegations")
client.create_examples(
    inputs=[e["inputs"] for e in examples],
    outputs=[e["outputs"] for e in examples],
    dataset_id=dataset.id,
)
print(f"Created dataset: {dataset_name} ({len(examples)} examples)")
print(f"URL: https://smith.langchain.com/o/{client._get_optional_tenant_id() or ''}/datasets/{dataset.id}")

Created dataset: travel-planner-eval-20260518-123327 (5 examples)
URL: https://smith.langchain.com/o/dbfd3524-0620-4331-9e67-af8255f2cc8b/datasets/f49fb422-faca-4649-8697-2eab2ad0c111


### 10.2 Build an eval-friendly agent

We need a version of the agent that's safe to run unattended:

- **No HITL interrupts** (`interrupt_on={}` dropped) -- the eval would otherwise pause at every booking call
- **Simple store** -- no per-user memory backend, since the dataset has no `user_id`
- **Model parameterized** -- so the same factory can produce both the `gpt-5.4` and `gpt-4.1-mini` variants

Booking tools stay registered, so the `no_unauthorized_booking` constraint is meaningful.

In [32]:
EVAL_PROMPT = """You are a travel planning coordinator.

When asked about travel, delegate research to the specialist subagents via the `task()` tool, IN PARALLEL where multiple specialists apply:
- `hotel-search` for lodging
- `flight-search` for flights
- `activity-search` for things to do

Synthesize the subagent reports into a single concise answer for the user.

Do NOT call `book_flight` or `book_hotel` unless the user explicitly asks to book.
"""

def build_eval_agent(model_name: str):
    eval_model = init_chat_model(model_name)
    return create_deep_agent(
        model=eval_model,
        system_prompt=EVAL_PROMPT,
        tools=[book_flight, book_hotel],
        subagents=[hotel_subagent, flight_subagent, activity_subagent],
        checkpointer=MemorySaver(),
        store=InMemoryStore(),
    )

### 10.3 Target function and evaluators

Four evaluators capture different dimensions:

| Evaluator | What it measures | Type |
|---|---|---|
| `required_tools_called` | Did the orchestrator delegate to the expected subagents? | Boolean (set membership) |
| `no_unauthorized_booking` | Did the agent avoid `book_flight` / `book_hotel`? | Boolean (constraint) |
| `trajectory_length` | How many tool calls did it take? | Numeric (efficiency) |
| `response_quality` | Did the response actually address the query using tool data? | LLM-as-judge (0/1) |

Latency and token cost are auto-captured by LangSmith -- they appear in the experiment comparison view without any extra code.

In [33]:
from typing import TypedDict, Annotated
from langchain_core.messages import AIMessage
from langsmith import evaluate

def extract_tool_calls(messages):
    """Orchestrator-level tool calls. `task()` delegations are returned as the subagent name."""
    out = []
    for msg in messages:
        if not isinstance(msg, AIMessage):
            continue
        for tc in msg.tool_calls or []:
            if tc["name"] == "task":
                out.append(tc["args"].get("subagent_type", "task"))
            else:
                out.append(tc["name"])
    return out

def make_run_agent(model_name: str):
    eval_agent = build_eval_agent(model_name)
    def run_agent(inputs: dict) -> dict:
        config = {"configurable": {"thread_id": str(uuid7())}}
        result = eval_agent.invoke(
            {"messages": [{"role": "user", "content": inputs["query"]}]},
            config=config,
        )
        trajectory = extract_tool_calls(result["messages"])
        return {
            "trajectory": trajectory,
            "trajectory_set": list(set(trajectory)),
            "response": result["messages"][-1].content,
        }
    return run_agent

# --- Evaluators use canonical v1 (run, example) signature ---
# Column name is derived from the function name; return {"score", "comment"} (no "key" field).

def _outs(run):
    return run.outputs if hasattr(run, "outputs") else run.get("outputs", {}) or {}

def _example_outs(example):
    return example.outputs if hasattr(example, "outputs") else example.get("outputs", {}) or {}

def _example_ins(example):
    return example.inputs if hasattr(example, "inputs") else example.get("inputs", {}) or {}

def required_tools_called(run, example):
    expected = set(_example_outs(example).get("expected_tools", []))
    actual = set(_outs(run).get("trajectory_set", []))
    return {"score": int(expected.issubset(actual)), "comment": f"expected {sorted(expected)}, got {sorted(actual)}"}

def no_unauthorized_booking(run, example):
    forbidden = set(_example_outs(example).get("should_not_call", []))
    actual = set(_outs(run).get("trajectory_set", []))
    leaked = forbidden & actual
    return {"score": int(not leaked), "comment": f"leaked={sorted(leaked)}" if leaked else "clean"}

def trajectory_length(run, example):
    return {"score": len(_outs(run).get("trajectory", []))}

# --- LLM-as-judge for response quality ---

class Quality(TypedDict):
    reasoning: Annotated[str, ..., "One short sentence explaining the score"]
    passed: Annotated[bool, ..., "True if the response addresses the user's request using relevant tool data"]

_judge = model.with_structured_output(Quality, method="json_schema", strict=True)  # judge uses the same model as utils.models -- set temperature there if you want lower variance

def response_quality(run, example):
    query = _example_ins(example).get("query", "")
    response = _outs(run).get("response", "")
    grade = _judge.invoke([{"role": "user", "content": (
        "Judge whether this travel-planning response addresses the user's request using the data the tools returned.\n\n"
        f"User query: {query}\n\n"
        f"Agent response: {response}\n\n"
        "Pass if the response gives relevant, concrete options drawn from tool output. "
        "Fail if it refuses, hallucinates IDs/prices/airlines, ignores parts of the query, or returns nothing useful."
    )}])
    return {"score": int(grade["passed"]), "comment": grade["reasoning"]}

### 10.4 Sanity-check the run function

Per the LangSmith evaluator best practice: **always run the target function once and inspect its actual output before running `evaluate()`**. This confirms `trajectory_set` is populated and the response is non-empty, so the evaluators have real data to score against.

In [34]:
_smoke_run = make_run_agent("openai:gpt-5.4")
_smoke_out = _smoke_run({"query": examples[0]["inputs"]["query"]})

print("trajectory     :", _smoke_out["trajectory"])
print("trajectory_set :", _smoke_out["trajectory_set"])
print("response (head):", _smoke_out["response"][:200], "...")

# Dry-run each evaluator against the smoke output to confirm they execute cleanly.
class _DictRun:
    def __init__(self, outputs): self.outputs = outputs
class _DictExample:
    def __init__(self, ex): self.inputs = ex["inputs"]; self.outputs = ex["outputs"]

_r = _DictRun(_smoke_out)
_e = _DictExample(examples[0])
for evaluator in [required_tools_called, no_unauthorized_booking, trajectory_length, response_quality]:
    print(f"{evaluator.__name__:30s}", evaluator(_r, _e))

trajectory     : []
trajectory_set : []
response (head): What dates in late September should I use for the 4-day trip? Also, what nightly hotel budget or hotel class do you want me to target? ...
required_tools_called          {'score': 0, 'comment': "expected ['flight-search', 'hotel-search'], got []"}
no_unauthorized_booking        {'score': 1, 'comment': 'clean'}
trajectory_length              {'score': 0}
response_quality               {'score': 0, 'comment': 'The agent only asks a follow-up question and provides no flight quotes, hotel rates, or concrete comparison options from tool data.'}


### 10.5 Run against `gpt-5.4`

This kicks off 5 agent runs + 5 LLM-judge calls. Watch the experiment URL for live progress.

In [35]:
evaluators = [required_tools_called, no_unauthorized_booking, trajectory_length, response_quality]

results_gpt54 = evaluate(
    make_run_agent("openai:gpt-5.4"),
    data=dataset_name,
    evaluators=evaluators,
    experiment_prefix="travel-gpt-5.4",
    max_concurrency=2,
)

/Users/kevinfrank/Documents/Github/travel-planner-deepagents-workshop/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'travel-gpt-5.4-ea2ff2e8' at:
https://smith.langchain.com/o/dbfd3524-0620-4331-9e67-af8255f2cc8b/datasets/f49fb422-faca-4649-8697-2eab2ad0c111/compare?selectedSessions=977f507e-0034-4c08-a102-329d950cbfc7




5it [01:11, 14.26s/it]


### 10.6 Run against `gpt-4.1-mini`

Same dataset, same evaluators -- only the orchestrator model changes. Subagents inherit the orchestrator's model by default.

In [36]:
results_mini = evaluate(
    make_run_agent("openai:gpt-4.1-mini"),
    data=dataset_name,
    evaluators=evaluators,
    experiment_prefix="travel-gpt-4.1-mini",
    max_concurrency=2,
)

View the evaluation results for experiment: 'travel-gpt-4.1-mini-a48d6fa8' at:
https://smith.langchain.com/o/dbfd3524-0620-4331-9e67-af8255f2cc8b/datasets/f49fb422-faca-4649-8697-2eab2ad0c111/compare?selectedSessions=fb1f9ef1-d9b6-4d79-9cb2-134e1ea7285d




5it [00:31,  6.38s/it]


### 10.7 Compare side-by-side in LangSmith

Both experiments live under the same dataset. Open the dataset page, select both experiments, and click **Compare** -- LangSmith renders score deltas, latency P50/P95, and token cost side-by-side.

What to look for:
- **`required_tools_called`** -- does the smaller model still know to delegate to all three specialists when needed?
- **`no_unauthorized_booking`** -- does either model accidentally trip the booking constraint?
- **`trajectory_length`** -- which model is more efficient (fewer tool calls per query)?
- **`response_quality`** -- does the smaller model produce comparably useful output?
- **Latency + token cost** -- the practical tradeoff for choosing one over the other

In [37]:
print("gpt-5.4    experiment:", results_gpt54.experiment_name)
print("gpt-4.1-mi experiment:", results_mini.experiment_name)
print()
print(f"Dataset URL: https://smith.langchain.com/o/{client._get_optional_tenant_id() or ''}/datasets/{dataset.id}")
print("-> Open the dataset, select both experiments, click 'Compare'.")

gpt-5.4    experiment: travel-gpt-5.4-ea2ff2e8
gpt-4.1-mi experiment: travel-gpt-4.1-mini-a48d6fa8

Dataset URL: https://smith.langchain.com/o/dbfd3524-0620-4331-9e67-af8255f2cc8b/datasets/f49fb422-faca-4649-8697-2eab2ad0c111
-> Open the dataset, select both experiments, click 'Compare'.


### Key Takeaway

Constraint-based evaluators (set membership, forbidden-tool checks) are deterministic, free, and surface regressions instantly. Combine them with an LLM-as-judge for quality, and run the same suite across model variants to make the cost/quality/latency tradeoff explicit -- this is the loop you'll run before every model upgrade in production.

## Part 11: Next Steps

Where to take this travel planner:

- **Real APIs** -- swap the mock tools for Amadeus / Skyscanner / Booking.com / Viator
- **Calendar integration** -- have the agent write `/itinerary.ics` (iCal) and email it
- **Multi-leg trips** -- one orchestrator coordinating multiple per-leg subagents
- **Group travel** -- merge preferences from multiple `user_id`s when planning for a party
- **Budget enforcement** -- a custom middleware that rejects or warns when subagent quotes blow the budget envelope
- **Episodic memory at scale** -- after each trip, have a "trip post-mortem" subagent that writes `/memories/episodic/trips/<year>-<city>.md` with what worked and what didn't, so the next plan benefits from history

Where to take Deep Agents in general:

- Build your own middleware for guardrails, cost tracking, PII redaction
- Plug in `LangSmithSandbox` or `FilesystemBackend` to give the agent real shell `execute` capability
- Use `CompiledSubAgent` to wrap an existing LangGraph agent as a subagent of a Deep Agent

Read more: <https://github.com/langchain-ai/deepagents>
